[Guide Index](README.md)


# 02.08 OWL 2 Manchester Syntax

Manchester Syntax is an English-readable notation for OWL 2 ontologies — `Class: Cat SubClassOf: Animal` reads more naturally than raw RDF triples:

| Manchester Syntax | Equivalent RDF (Turtle) |
|---|---|
| `Class: Cat SubClassOf: Animal` | `:Cat rdf:type owl:Class . :Cat rdfs:subClassOf :Animal .` |

`StarLayer` is able to parse Manchester text into its RDF equivalent, and serialize an RDF graph back to Manchester text. `StarLayerGraph` operates over RDF graphs, so an ontology expressed as Manchester Syntax needs to be translated into a graph before it can be used with `StarLayerGraph`; for example, as an ontology to support OWL-based reasoning (see 02.06 and 02.07).

This guide discusses Manchester Syntax to RDF (section 02.08.1), RDF back to Manchester Syntax (section 02.08.2), reasoning over the ontology (section 02.08.3), and the error for text that is not Manchester Syntax (section 02.08.4). [Appendix C](appendix-c-manchester-syntax-reference.ipynb) goes through the notation construct by construct, each with its RDF equivalent.

A Manchester Syntax document's own structure - frames, clauses, expressions - can also be treated as an inspectable, editable RDF graph. That is covered in the [OWL Manchester Syntax as RDF guide](06.05-manchester-syntax-as-rdf.ipynb).

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01.01-getting-started.ipynb). No extra runtime dependencies are needed - Manchester Syntax parsing is pure Python.

Run cells from top to bottom; later cells reuse variables from earlier ones.

A `StarLayerGraph` starts with no prefix bindings, so the setup cell sets default prefixes (`:`, `owl:`, `rdfs:`) that every new graph starts with; `serialize(format="turtle", pretty=True)` then writes grouped, nested Turtle with only the prefixes that are used. (`xsd:` is added by the Manchester parser when a document uses it.)


In [2]:
from starlayer.graph import Namespace, OWL, RDFS, StarLayerGraph, set_default_prefixes

EX = Namespace("http://example.org/vehicles#")

# every graph made after this call starts with these prefixes bound
set_default_prefixes({"": EX, "owl": OWL, "rdfs": RDFS})


def show(manchester_text):
    """Parse a Manchester Syntax snippet and print the RDF it maps to."""
    g = StarLayerGraph()
    g.parse(data=f"Prefix: : <{EX}>\n" + manchester_text, format="manchester")
    print(g.serialize(format="turtle", pretty=True))
    return g

## 02.08.1 Manchester Syntax to RDF

`g.parse(data=..., format="manchester")` reads Manchester Syntax text into a graph as OWL RDF. Each frame (`Class:`, `ObjectProperty:`, `Individual:` etc.) becomes the triples for the axioms in its clauses. A `Prefix:` line gives the text its names using the default prefix `:`.

Below, a small ontology is parsed into a graph and the RDF is printed as Turtle.

In [3]:
ontology = f"Prefix: : <{EX}>\n" + """
Class: Animal
Class: Leg
Class: Cat
    SubClassOf: Animal
    SubClassOf: hasLeg some Leg
    Annotations: rdfs:label "Cat"
ObjectProperty: hasLeg
"""

g = StarLayerGraph()
g.parse(data=ontology, format="manchester")
print(g.serialize(format="turtle", pretty=True))

@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Animal a owl:Class .

:Cat a owl:Class ;
    rdfs:label "Cat" ;
    rdfs:subClassOf :Animal, [
        a owl:Restriction ;
        owl:onProperty :hasLeg ;
        owl:someValuesFrom :Leg
    ] .

:Leg a owl:Class .

:hasLeg a owl:ObjectProperty .



## 02.08.2 RDF to Manchester Syntax

`g.serialize(format="manchester")` writes a graph as Manchester Syntax text, the reverse direction. Some axioms are returned in their canonical expanded form so they may not match the original input identically. However, the precise meaning is retained.

The graph is ordinary RDF, so it can be edited with standard graph operations before it is written back.

Below, the graph from section 02.08.1 is written as Manchester Syntax. Then an edit is written as Turtle, parsed into a small graph, and added to `g`: a cat has some owner who is a person. The graph is written again, and the new axiom appears in the Manchester text.

In [4]:
print(g.serialize(format="manchester"))

# the edit, written as Turtle: Cat SubClassOf hasOwner some Person
edit = StarLayerGraph()
edit.parse(data="""
@prefix : <http://example.org/vehicles#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Person a owl:Class .
:hasOwner a owl:ObjectProperty .
:Cat rdfs:subClassOf [
    a owl:Restriction ;
    owl:onProperty :hasOwner ;
    owl:someValuesFrom :Person
] .
""", format="turtle")

# add the edit's triples to g
g += edit

print("after the edit:")
print(g.serialize(format="manchester"))

Prefix: : <http://example.org/vehicles#>
Prefix: rdfs: <http://www.w3.org/2000/01/rdf-schema#>

Class: :Animal

Class: :Cat
    SubClassOf: :hasLeg some :Leg, :Animal
    Annotations: rdfs:label "Cat"

Class: :Leg

ObjectProperty: :hasLeg

after the edit:
Prefix: : <http://example.org/vehicles#>
Prefix: rdfs: <http://www.w3.org/2000/01/rdf-schema#>

Class: :Animal

Class: :Cat
    SubClassOf: :hasOwner some :Person, :hasLeg some :Leg, :Animal
    Annotations: rdfs:label "Cat"

Class: :Leg

Class: :Person

ObjectProperty: :hasLeg

ObjectProperty: :hasOwner



## 02.08.3 Reasoning over the ontology

OWL ontologies are often used to perform graph reasoning. OWL ontologies expressed in Manchester syntax need to be parsed to RDF before being used in graph inferencing, as discussed in [02.06](02.06-graphs-inferencing.ipynb) (RDF, RDFS and OWL 2 RL) and [02.07](02.07-owl-dl-reasoning.ipynb) (OWL 2 DL). The reasoner can use the axioms of the ontology to draw conclusions that the graph never explicitly states.

Below, `FamilyVehicle` is defined as a `Car` or a `Truck`, and `isEngineOf` is the inverse of `hasEngine`. `mycar` is a `Car` with an engine.

OWL 2 RL reasoning (`ENTAILMENT['OWL-RDF-Based']`) concludes that `mycar` is a `FamilyVehicle`, because it is a `Car`, and that `engine1` `isEngineOf` `mycar`.

In [8]:
from starlayer.graph import ENTAILMENT, RDF

vehicles = StarLayerGraph()
vehicles.parse(data=f"Prefix: : <{EX}>\n" + """
Class: Car
Class: Truck
Class: FamilyVehicle
    EquivalentTo: Car or Truck
ObjectProperty: hasEngine
ObjectProperty: isEngineOf
    InverseOf: hasEngine
Individual: engine1
Individual: mycar
    Types: Car
    Facts: hasEngine engine1
""", format="manchester")

name = "OWL-RDF-Based"
closed = vehicles.infer(entailment=ENTAILMENT[name])
print(f"{name}: mycar is a FamilyVehicle: {(EX.mycar, RDF.type, EX.FamilyVehicle) in closed}; "
      f"engine1 is an engine of mycar: {(EX.engine1, EX.isEngineOf, EX.mycar) in closed}")

OWL-RDF-Based: mycar is a FamilyVehicle: True; engine1 is an engine of mycar: True


## 02.08.4 Manchester syntax errors

Text that is not Manchester Syntax and cannot be parsed into RDF raises a `FormatError` (kind `"manchester"`; `why` is the reason and `line` the line where it is known).

Some tools such as Protégé and the OWL API that accept Manchester Syntax also accept SWRL rules. StarLayer does not support SWRL rules and text that includes rule frames is rejected.

For rules support, see the [SRL guide](04.01-srl-sparql-rules.ipynb).

Below, a `Rule:` and a one-member `EquivalentClasses:` are each rejected with a `FormatError`.

In [6]:
from starlayer.errors import FormatError

try:
    show("""
    Rule: Person(?x)
    """)
except FormatError as e:
    print(f"Rejected as expected: {e.why}")

try:
    show("""
    Class: A
    Class: B
    EquivalentClasses: A
    """)
except FormatError as e:
    print(f"Rejected as expected: {e.why}")


Rejected as expected: SWRL rules ('Rule:') are not supported by StarLayer: the W3C Manchester Syntax has no rules, they are an extension of some tools (Protege, the OWL API). Remove the rule, or use SRL (SPARQL Rules) for rules
Rejected as expected: EquivalentClasses: needs at least two members, got 1
